In [ ]:
# Copyright (c) 2015-present, Facebook, Inc.
# All rights reserved.
#
# This source code is licensed under the CC-by-NC license found in the
# LICENSE file in the root directory of this source tree.
#
import os
from os.path import join

import argparse
from bert_score import BERTScorer
import csv
import datasets
import json
import numpy as np
import time
import torch
from transformers import AutoTokenizer, pipeline
import transformers
import textattack
from textattack.attack_recipes import BERTAttackLi2020, BAEGarg2019, CLARE2020
from textattack.datasets import HuggingFaceDataset
from textattack.models.wrappers import ModelWrapper
from textattack.commands.attack.attack_args import HUGGINGFACE_DATASET_BY_MODEL
from textattack.constraints.overlap import MaxWordsPerturbed
from textattack.constraints.pre_transformation import RepeatModification, StopwordModification
from textattack.constraints.semantics.sentence_encoders import UniversalSentenceEncoder
from textattack.goal_functions import UntargetedClassification
from textattack.search_methods import GreedyWordSwapWIR
from textattack.shared.attack import Attack
from textattack.transformations import WordSwapMaskedLM
from textattack.attack_recipes.attack_recipe import AttackRecipe
from textattack.goal_functions.classification.targeted_classification import TargetedClassification
from textattack.shared import AttackedText
from textattack.constraints.grammaticality import PartOfSpeech
from textattack.constraints.overlap import MaxWordsPerturbed
from textattack.constraints.pre_transformation import (
    RepeatModification,
    StopwordModification,
)
from textattack.constraints.semantics import WordEmbeddingDistance
from textattack.goal_functions import UntargetedClassification
from textattack.search_methods import BeamSearch
from textattack.shared.attack import Attack
from textattack.transformations import WordSwapGradientBased

from textattack.attack_results import SuccessfulAttackResult, FailedAttackResult

from src.attacks import build_baegarg2019, build_attack, USE

In [ ]:

dataset_name = ""
assert dataset_name in ['dbpedia14', 'sst2', 'ag_news', 'yelp_polarity', 'stanfordnlp/imdb']
data_folder = ""
batch_size = 32
dump_path = ""
model_name = "bert-base-uncased"
target_dir = ""
chunk_id = int
chunk_size = int
attack_name = "
assert attack_name in ["bae", "bert-attack", "custom"]
bae_threshold = 0.8
query_budget = None
radioactive = bool
targeted = True
ckpt = str





In [ ]:
def load_data(dataset_name, data_folder):
    if dataset_name == "dbpedia14":
        dataset = load_dataset("csv", column_names=["label", "title", "sentence"],
                                data_files={"train": os.path.join(data_folder, "dbpedia_csv/train.csv"),
                                            "validation": os.path.join(data_folder, "dbpedia_csv/test.csv")})
        dataset = dataset.map(target_offset, batched=True)
        num_labels = 14
    elif dataset_name == "ag_news":
        dataset = load_dataset("ag_news")
        num_labels = 4
    elif dataset_name == "imdb":
        dataset = load_dataset("stanfordnlp/imdb")
        num_labels = 2
    elif dataset_name == "yelp":
        dataset = load_dataset("yelp_polarity")
        num_labels = 2
    elif dataset_name == "mnli":
        dataset = load_dataset("nyu-mll/glue")
        num_labels = 3
    dataset = dataset.shuffle(seed=0)

    return dataset, num_labels

In [ ]:
def build_model_name(dataset_name, model_name):
  model_prefix = ""
  dataset_appendix = ""
  if dataset_name =
  ag-news
  imdb
  CoLA
  MRPC
  QQP
  RTE
  SST-2
  STS-B
  WNLI
  rotten-tomatoes
  snli

  albert-base-v2
  bert-base-uncased
  cnn
  distilbert-base-uncased
  lstm
  roberta-base
  xlnet-base-cased
  full_name = f"textattack/{model_prefix}-{dataset_appendix}"
  return full_name


In [ ]:
# Loading data
dataset, num_labels = load_data(params)
dataset = dataset["train"]
text_key = 'text'
if params.dataset == "dbpedia14":
    text_key = 'content'
print(f"Loaded dataset {params.dataset}, that has {len(dataset)} rows")

# Load model and tokenizer from HuggingFace
model_class = transformers.AutoModelForSequenceClassification
model = model_class.from_pretrained(model_name, num_labels=num_labels).cuda()

if ckpt != None:
    state_dict = torch.load(ckpt)
    model.load_state_dict(state_dict)
tokenizer = textattack.models.tokenizers.AutoTokenizer(model_name)
model_wrapper = textattack.models.wrappers.HuggingFaceModelWrapper(model, tokenizer, batch_size=params.batch_size)

# Create radioactive directions and modify classification layer to use those
if radioactive:
    torch.manual_seed(0)
    radioactive_directions = torch.randn(num_labels, 768)
    radioactive_directions /= torch.norm(radioactive_directions, dim=1, keepdim=True)
    print(radioactive_directions)
    model.classifier.weight.data = radioactive_directions.cuda()
    model.classifier.bias.data = torch.zeros(num_labels).cuda()

start_index = target_dir * chunk_size
end_index  = start_index + chunk_size

if target_dir is not None:
    target_file = join(target_dir, f"{target_dir}.csv")
    f = open(target_file, "w")
    f = csv.writer(f, delimiter=',', quotechar='"', quoting=csv.QUOTE_NONNUMERIC)

# Creating attack
print(f"Building {attack_name} attack")
if attack_name == "custom":
    current_label = -1
    if targeted:
        current_label = dataset[start_index]['label']
        assert all([dataset[i]['label'] == current_label for i in range(start_index, end_index)])
    attack = build_attack(model_wrapper, current_label)
elif attack_name == "bae":
    print(f"Building BAE method with threshold={bae_threshold:.2f}")
    attack = build_baegarg2019(model_wrapper, threshold_cosine=bae_threshold, query_budget=query_budget)
elif attack_name == "bert-attack":
    assert query_budget is None
    attack = BERTAttackLi2020.build(model_wrapper)
elif attack_name == "clare":
    assert query_budget is None
    attack = CLARE2020.build(model_wrapper)

# Launching attack
begin_time = time.time()
samples = [(dataset[i][text_key], attack.goal_function.get_output(AttackedText(dataset[i][text_key]))) for i in range(start_index, end_index)]
results = list(attack.attack_dataset(samples))

# Storing attacked text
bert_scorer = BERTScorer(model_type="bert-base-uncased", idf=False)

n_success = 0
similarities = []
queries = []
use = USE()

for i_result, result in enumerate(results):
    print("")
    print(50 * "*")
    print("")
    text = dataset[start_index + i_result][text_key]
    ptext = result.perturbed_text()
    i_data = start_index + i_result
    if target_dir is not None:
        if params.dataset == 'dbpedia14':
            f.writerow([dataset[i_data]['label'] + 1, dataset[i_data]['title'], ptext])
        else:
            f.writerow([dataset[i_data]['label'] + 1, ptext])

    print("True label ", dataset[i_data]['label'])
    print(f"CLEAN TEXT\n {text}")
    print(f"ADV TEXT\n {ptext}")

    if type(result) not in [SuccessfulAttackResult, FailedAttackResult]:
        print("WARNING: Attack neither succeeded nor failed...")
    print(result.goal_function_result_str())
    precision, recall, f1 = [r.item() for r in bert_scorer.score([ptext], [text])]
    print(f"Bert scores: precision {precision:.2f}, recall: {recall:.2f}, f1: {f1:.2f}")
    initial_logits = model_wrapper([text])
    final_logits = model_wrapper([ptext])
    print("Initial logits", initial_logits)
    print("Final logits", final_logits)
    print("Logits difference", final_logits - initial_logits)

    # Statistics
    n_success += 1 if type(result) is SuccessfulAttackResult else 0
    queries.append(result.num_queries)
    similarities.append(use.compute_sim([text], [ptext]))

print("Processing all samples took %.2f" % (time.time() - begin_time))
print(f"Total success: {n_success}/{len(results)}")
logs = {
    "success_rate": n_success / len(results),
    "avg_queries": sum(queries) / len(queries),
    "queries": queries,
    "avg_similarity": sum(similarities) / len(similarities),
    "similarities": similarities,
}
print("__logs:" + json.dumps(logs))
if target_dir is not None:
    f.close()


if __name__ == "__main__":
print("Using text attack from ", textattack.__file__)
# Parse arguments
parser = get_parser()
params = parser.parse_args()
# if not radioactive:
#     assert ckpt is not None, "Should specify --ckpt if not radioactive."
assert not (radioactive and not targeted), "Radioactive means targeted"

# Run main code
begin_time = time.time()
main(params)
print("Running program took %.2f" % (time.time() - begin_time))
